In [ ]:
# Install RewardBench package
!pip install rewardbench

# Remove old clone if it exists
!rm -rf reward-bench

# Clone official RewardBench repository for scripts/run_rm.py and scripts/run_dpo.py
!git clone https://github.com/allenai/reward-bench.git

# Move into RewardBench repo
%cd reward-bench

In [ ]:
# Check GPU availability
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
# Create folders to save experiment logs
import os

os.makedirs("/content/rewardbench_results/raw_outputs", exist_ok=True)
os.makedirs("/content/rewardbench_results/figures", exist_ok=True)

print("Folders created.")

In [ ]:
# Reduce unnecessary logging/noise
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

print("Environment variables set.")

In [ ]:
# Patch RewardBench scripts so they do not upload results to Hugging Face
from pathlib import Path

for script_path in ["scripts/run_rm.py", "scripts/run_dpo.py"]:
    path = Path(script_path)
    text = path.read_text()

    patch = """
# Course project patch: disable Hugging Face Hub upload in Colab
def save_to_hub(*args, **kwargs):
    print("Skipping Hugging Face Hub upload; results kept locally.")
    return "local_only"

"""

    if "Course project patch: disable Hugging Face Hub upload" not in text:
        text = text.replace("\ndef main():", "\n" + patch + "\ndef main():")
        path.write_text(text)
        print(f"Patched {script_path}")
    else:
        print(f"Already patched {script_path}")

In [ ]:
# Run lightweight classifier-based reward model
!python scripts/run_rm.py \
  --model=OpenAssistant/reward-model-deberta-v3-large-v2 \
  --chat_template=raw \
  --batch_size=16 \
  2>&1 | tee /content/rewardbench_results/raw_outputs/deberta_rm.log

In [ ]:
# Save completed DeBERTa results locally as CSV files

import pandas as pd
from pathlib import Path

# Create results folder
results_dir = Path("/content/rewardbench_results")
results_dir.mkdir(parents=True, exist_ok=True)

# Main model-level result
model_results = pd.DataFrame([
    {
        "model": "OpenAssistant/reward-model-deberta-v3-large-v2",
        "type": "Classifier RM",
        "overall": "",
        "chat": 0.8044692737430168,
        "chat_hard": 0.4407894736842105,
        "safety": 0.7662162162162162,
        "reasoning": 0.37757816336552624,
        "notes": "Completed local RewardBench run; HF upload disabled."
    }
])

# Long-format section scores
section_scores = pd.DataFrame([
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "section": "Chat", "score": 0.8044692737430168},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "section": "Chat Hard", "score": 0.4407894736842105},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "section": "Safety", "score": 0.7662162162162162},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "section": "Reasoning", "score": 0.37757816336552624},
])

# Detailed subset scores for deeper analysis
subset_scores = pd.DataFrame([
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "alpacaeval-easy", "score": 0.84},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "alpacaeval-hard", "score": 0.968421052631579},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "alpacaeval-length", "score": 0.6842105263157895},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "donotanswer", "score": 0.5147058823529411},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "hep-cpp", "score": 0.5975609756097561},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "hep-go", "score": 0.5853658536585366},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "hep-java", "score": 0.823170731707317},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "hep-js", "score": 0.4573170731707317},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "hep-python", "score": 0.573170731707317},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "hep-rust", "score": 0.7560975609756098},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "llmbar-adver-GPTInst", "score": 0.7065217391304348},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "llmbar-adver-GPTOut", "score": 0.0},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "llmbar-adver-manual", "score": 0.3695652173913043},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "llmbar-adver-neighbor", "score": 0.4925373134328358},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "llmbar-natural", "score": 0.21},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "math-prm", "score": 0.12304250559284116},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "mt-bench-easy", "score": 0.5},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "mt-bench-hard", "score": 0.8648648648648649},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "mt-bench-med", "score": 0.825},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "refusals-dangerous", "score": 0.64},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "refusals-offensive", "score": 0.7},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "xstest-should-refuse", "score": 0.948051948051948},
    {"model": "OpenAssistant/reward-model-deberta-v3-large-v2", "subset": "xstest-should-respond", "score": 0.868},
])

# Save CSV files
model_results.to_csv(results_dir / "model_comparison.csv", index=False)
section_scores.to_csv(results_dir / "section_scores.csv", index=False)
subset_scores.to_csv(results_dir / "subset_scores.csv", index=False)

print("Saved:")
print(results_dir / "model_comparison.csv")
print(results_dir / "section_scores.csv")
print(results_dir / "subset_scores.csv")

In [ ]:
# Check saved result files
!ls -lh /content/rewardbench_results

In [ ]:
# Display saved DeBERTa result
import pandas as pd

pd.read_csv("/content/rewardbench_results/model_comparison.csv")

In [ ]:
# Run Qwen DPO with smallest batch size to avoid memory error
!python scripts/run_dpo.py \
  --model=Qwen/Qwen1.5-0.5B-Chat \
  --ref_model=Qwen/Qwen1.5-0.5B \
  --batch_size=1 \
  2>&1 | tee /content/rewardbench_results/raw_outputs/qwen_05b_dpo_bs1.log

## Part 2: Per-Instance Analysis
Captures per-example predictions from a targeted subset of RewardBench datasets
to enable qualitative comparison and pairwise disagreement analysis between models.

In [ ]:
# Load the RewardBench dataset from HuggingFace
from datasets import load_dataset
import pandas as pd
import torch
import os

os.makedirs('/content/rewardbench_results/per_instance', exist_ok=True)

# Load full filtered split
dataset = load_dataset('allenai/reward-bench', split='filtered')

# Focus on four diagnostic subsets
target_subsets = ['hep-python', 'llmbar-adver-GPTOut', 'llmbar-natural', 'math-prm']
dataset_filtered = dataset.filter(lambda x: x['subset'] in target_subsets)

print(f'Total examples across target subsets: {len(dataset_filtered)}')
pd.Series(dataset_filtered['subset']).value_counts()

In [ ]:
# ── DeBERTa (Classifier) ──────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification

deberta_name = 'OpenAssistant/reward-model-deberta-v3-large-v2'
deb_tok = AutoTokenizer.from_pretrained(deberta_name)
deb_model = AutoModelForSequenceClassification.from_pretrained(deberta_name).cuda()
deb_model.eval()

rows = []
for ex in dataset_filtered:
    with torch.no_grad():
        enc_c = deb_tok(ex['prompt'], ex['chosen'],   return_tensors='pt',
                        truncation=True, max_length=512).to('cuda')
        enc_r = deb_tok(ex['prompt'], ex['rejected'], return_tensors='pt',
                        truncation=True, max_length=512).to('cuda')
        sc = deb_model(**enc_c).logits[0].item()
        sr = deb_model(**enc_r).logits[0].item()
    rows.append({
        'subset':   ex['subset'],
        'prompt':   ex['prompt'],
        'chosen':   ex['chosen'],
        'rejected': ex['rejected'],
        'deberta_chosen_score':   sc,
        'deberta_rejected_score': sr,
        'deberta_correct': sc > sr,
    })

df = pd.DataFrame(rows)
print('DeBERTa accuracy per subset:')
print(df.groupby('subset')['deberta_correct'].mean().round(3))

# Free memory before next model
del deb_model
torch.cuda.empty_cache()

In [ ]:
# ── ArmoRM (Custom Classifier) ───────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

armo_name = 'RLHFlow/ArmoRM-Llama3-8B-v0.1'
armo_tok  = AutoTokenizer.from_pretrained(armo_name, trust_remote_code=True)
armo_model = AutoModelForSequenceClassification.from_pretrained(
    armo_name, trust_remote_code=True,
    torch_dtype=torch.bfloat16, num_labels=1
).cuda()
armo_model.eval()

def armo_score(prompt, response):
    messages = [{'role': 'user', 'content': prompt},
                {'role': 'assistant', 'content': response}]
    text = armo_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    ids  = armo_tok(text, return_tensors='pt', truncation=True, max_length=1024).to('cuda')
    with torch.no_grad():
        out = armo_model(**ids)
    return out.logits[0].item()

armo_chosen, armo_rejected, armo_correct = [], [], []
for ex in dataset_filtered:
    sc = armo_score(ex['prompt'], ex['chosen'])
    sr = armo_score(ex['prompt'], ex['rejected'])
    armo_chosen.append(sc)
    armo_rejected.append(sr)
    armo_correct.append(sc > sr)

df['armo_chosen_score']   = armo_chosen
df['armo_rejected_score'] = armo_rejected
df['armo_correct']        = armo_correct

print('ArmoRM accuracy per subset:')
print(df.groupby('subset')['armo_correct'].mean().round(3))

del armo_model
torch.cuda.empty_cache()

In [ ]:
# Force full GPU memory release
import gc, torch

try: del armo_model
except: pass
try: del pol_model
except: pass
try: del ref_model
except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"Free GPU memory: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# ── Zephyr (DPO) ──────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

pol_name = 'HuggingFaceH4/zephyr-7b-beta'
ref_name = 'HuggingFaceH4/mistral-7b-sft-beta'

zep_tok   = AutoTokenizer.from_pretrained(pol_name)
pol_model = AutoModelForCausalLM.from_pretrained(pol_name, torch_dtype=torch.float16).cuda()
ref_model = AutoModelForCausalLM.from_pretrained(ref_name, torch_dtype=torch.float16).cuda()
pol_model.eval(); ref_model.eval()

def dpo_implicit_reward(prompt, response, policy, reference, tokenizer, beta=0.1):
    """Compute beta * (log pi_pol - log pi_ref) averaged over response tokens."""
    full_text = prompt + '\n' + response
    ids = tokenizer(full_text, return_tensors='pt',
                    truncation=True, max_length=512).to('cuda')
    with torch.no_grad():
        pol_logits = policy(**ids).logits
        ref_logits = reference(**ids).logits
    # Shift: predict token i+1 from position i
    labels = ids['input_ids'][:, 1:]
    pol_lp = pol_logits[:, :-1, :].log_softmax(-1)
    ref_lp = ref_logits[:, :-1, :].log_softmax(-1)
    pol_token_lp = pol_lp.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    ref_token_lp = ref_lp.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    return beta * (pol_token_lp - ref_token_lp).mean().item()

zep_chosen, zep_rejected, zep_correct = [], [], []
for ex in dataset_filtered:
    sc = dpo_implicit_reward(ex['prompt'], ex['chosen'],   pol_model, ref_model, zep_tok)
    sr = dpo_implicit_reward(ex['prompt'], ex['rejected'], pol_model, ref_model, zep_tok)
    zep_chosen.append(sc)
    zep_rejected.append(sr)
    zep_correct.append(sc > sr)

df['zephyr_chosen_score']   = zep_chosen
df['zephyr_rejected_score'] = zep_rejected
df['zephyr_correct']        = zep_correct

print('Zephyr accuracy per subset:')
print(df.groupby('subset')['zephyr_correct'].mean().round(3))

del pol_model, ref_model
torch.cuda.empty_cache()

In [ ]:
# Force full GPU memory release
import gc, torch

try: del armo_model
except: pass
try: del pol_model
except: pass
try: del ref_model
except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"Free GPU memory: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# ── Qwen (DPO) ────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

pol_name = 'Qwen/Qwen1.5-0.5B-Chat'
ref_name = 'Qwen/Qwen1.5-0.5B'

qwen_tok   = AutoTokenizer.from_pretrained(pol_name)
pol_model  = AutoModelForCausalLM.from_pretrained(pol_name, torch_dtype=torch.float16).cuda()
ref_model  = AutoModelForCausalLM.from_pretrained(ref_name, torch_dtype=torch.float16).cuda()
pol_model.eval(); ref_model.eval()

qwen_chosen, qwen_rejected, qwen_correct = [], [], []
for ex in dataset_filtered:
    sc = dpo_implicit_reward(ex['prompt'], ex['chosen'],   pol_model, ref_model, qwen_tok)
    sr = dpo_implicit_reward(ex['prompt'], ex['rejected'], pol_model, ref_model, qwen_tok)
    qwen_chosen.append(sc)
    qwen_rejected.append(sr)
    qwen_correct.append(sc > sr)

df['qwen_chosen_score']   = qwen_chosen
df['qwen_rejected_score'] = qwen_rejected
df['qwen_correct']        = qwen_correct

print('Qwen accuracy per subset:')
print(df.groupby('subset')['qwen_correct'].mean().round(3))

del pol_model, ref_model
torch.cuda.empty_cache()

In [ ]:
# ── Save per-instance CSV ─────────────────────────────────────────────────────
df.to_csv('/content/rewardbench_results/per_instance/per_instance_results.csv', index=False)
print(f'Saved {len(df)} rows to per_instance_results.csv')
df[['subset','deberta_correct','armo_correct','zephyr_correct','qwen_correct']].head(10)

In [ ]:
# ── Pairwise disagreement table ───────────────────────────────────────────────
# Count cases where model X is correct and model Y is wrong
import pandas as pd

model_cols   = ['deberta_correct', 'armo_correct', 'zephyr_correct', 'qwen_correct']
model_labels = ['DeBERTa', 'ArmoRM', 'Zephyr', 'Qwen']

rows = []
for col_x, lx in zip(model_cols, model_labels):
    for col_y, ly in zip(model_cols, model_labels):
        if col_x == col_y:
            count = '-'
        else:
            count = int(((df[col_x] == True) & (df[col_y] == False)).sum())
        rows.append({'X \ Y': lx, ly: count})

pivot = pd.DataFrame(rows).groupby('X \ Y').first().reset_index()
pivot = pivot.set_index('X \ Y')[model_labels]
print('Pairwise disagreement: row = X correct, column = Y wrong')
print(pivot.to_string())

pivot.to_csv('/content/rewardbench_results/per_instance/pairwise_disagreement.csv')
print('\nSaved pairwise_disagreement.csv')

In [ ]:
# ── Qualitative examples: ArmoRM correct, DeBERTa wrong ──────────────────────
# Find the most interesting disagreement cases in hep-python and llmbar-adver-GPTOut

for subset in ['hep-python', 'llmbar-adver-GPTOut']:
    cases = df[
        (df['subset'] == subset) &
        (df['armo_correct'] == True) &
        (df['deberta_correct'] == False)
    ].head(2)

    print(f'\n=== {subset}: ArmoRM correct, DeBERTa wrong ({len(cases)} shown) ===')
    for i, row in cases.iterrows():
        print(f'\n--- Example {i} ---')
        print(f'PROMPT:   {row["prompt"][:300]}')
        print(f'CHOSEN:   {row["chosen"][:300]}')
        print(f'REJECTED: {row["rejected"][:300]}')
        print(f'ArmoRM  chosen={row["armo_chosen_score"]:.3f}  rejected={row["armo_rejected_score"]:.3f}')
        print(f'DeBERTa chosen={row["deberta_chosen_score"]:.3f}  rejected={row["deberta_rejected_score"]:.3f}')

print('\n=== Reverse: DeBERTa correct, ArmoRM wrong ===')
reverse = df[(df['deberta_correct'] == True) & (df['armo_correct'] == False)]
print(f'Total reverse cases: {len(reverse)}')
if len(reverse) > 0:
    row = reverse.iloc[0]
    print(f'Subset: {row["subset"]}')
    print(f'PROMPT:   {row["prompt"][:300]}')
    print(f'CHOSEN:   {row["chosen"][:300]}')
    print(f'REJECTED: {row["rejected"][:300]}')
    print(f'ArmoRM  chosen={row["armo_chosen_score"]:.3f}  rejected={row["armo_rejected_score"]:.3f}')
    print(f'DeBERTa chosen={row["deberta_chosen_score"]:.3f}  rejected={row["deberta_rejected_score"]:.3f}')

In [ ]:
# ── Download results ──────────────────────────────────────────────────────────
from google.colab import files

files.download('/content/rewardbench_results/per_instance/per_instance_results.csv')
files.download('/content/rewardbench_results/per_instance/pairwise_disagreement.csv')
print('Downloaded both CSVs.')

Generate Heat Map

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Load pairwise disagreement CSV
pairwise = pd.read_csv('/content/rewardbench_results/per_instance/pairwise_disagreement.csv', index_col=0)

model_labels = ['DeBERTa', 'ArmoRM', 'Zephyr', 'Qwen']

# Convert to numeric, replace '-' diagonal with NaN
matrix = pairwise.copy().replace('-', np.nan).astype(float)

fig, ax = plt.subplots(figsize=(7, 5.5))

# Plot heatmap — mask diagonal
masked = np.ma.masked_where(np.eye(4, dtype=bool), matrix.values)
im = ax.imshow(masked, cmap='Blues', aspect='auto')

# Diagonal in light gray
diag_mask = np.where(np.eye(4, dtype=bool), 0, np.nan)
ax.imshow(diag_mask, cmap='Greys', aspect='auto', vmin=0, vmax=1, alpha=0.15)

# Colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Number of examples\n(X correct, Y wrong)', fontsize=10)

# Annotate cells
for i in range(4):
    for j in range(4):
        if i == j:
            ax.text(j, i, '—', ha='center', va='center', fontsize=13, color='gray')
        else:
            val = matrix.values[i, j]
            if np.isnan(val):
                continue
            thresh = masked.max() * 0.6
            color  = 'white' if val > thresh else 'black'
            ax.text(j, i, str(int(val)), ha='center', va='center',
                    fontsize=13, fontweight='bold', color=color)

ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(model_labels, fontsize=11)
ax.set_yticklabels(model_labels, fontsize=11)
ax.set_xlabel('Y  (wrong)', fontsize=12, labelpad=8)
ax.set_ylabel('X  (correct)', fontsize=12, labelpad=8)
ax.set_title('Pairwise Disagreement: X correct when Y wrong', fontsize=12, pad=12)

plt.tight_layout()
plt.savefig('/content/rewardbench_results/figures/pairwise_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved pairwise_heatmap.png')

Generate Radar Chart

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Section scores from your existing results
sections = ['Chat', 'Chat Hard', 'Safety', 'Reasoning']
models = {
    'ArmoRM 8B':  [0.972, 0.763, 0.904, 0.974],
    'Zephyr 7B':  [0.925, 0.660, 0.630, 0.761],
    'DeBERTa RM': [0.805, 0.441, 0.766, 0.378],
    'Qwen 0.5B':  [0.366, 0.632, 0.566, 0.607],
}
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

# Number of variables
N = len(sections)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for (model, scores), color in zip(models.items(), colors):
    values = scores + scores[:1]  # close polygon
    ax.plot(angles, values, 'o-', linewidth=2, label=model, color=color)
    ax.fill(angles, values, alpha=0.08, color=color)

# Axis labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(sections, fontsize=12)
ax.set_ylim(0, 1.0)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8, color='gray')
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.xaxis.grid(True, linestyle='--', alpha=0.3)

ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
ax.set_title('RewardBench Section Scores by Model', fontsize=12, pad=20)

plt.tight_layout()
plt.savefig('/content/rewardbench_results/figures/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved radar_chart.png')

Score Margin Histogram:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load per-instance results
df = pd.read_csv('/content/rewardbench_results/per_instance/per_instance_results.csv')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)

models = [
    ('DeBERTa', 'deberta_chosen_score', 'deberta_rejected_score', 'deberta_correct', '#FF9800'),
    ('ArmoRM',  'armo_chosen_score',    'armo_rejected_score',    'armo_correct',    '#2196F3'),
]

for ax, (name, chosen_col, rejected_col, correct_col, color) in zip(axes, models):
    df['margin'] = df[chosen_col] - df[rejected_col]

    correct   = df[df[correct_col] == True]['margin']
    incorrect = df[df[correct_col] == False]['margin']

    bins = np.linspace(df['margin'].min(), df['margin'].max(), 35)

    ax.hist(correct,   bins=bins, alpha=0.65, label='Correct',   color=color,   edgecolor='white')
    ax.hist(incorrect, bins=bins, alpha=0.65, label='Incorrect', color='#E91E63', edgecolor='white')

    ax.axvline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.6)
    ax.set_title(f'{name}: Score Margin Distribution', fontsize=11)
    ax.set_xlabel('Chosen score − Rejected score', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Reward Score Margins: Correct vs Incorrect Predictions', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('/content/rewardbench_results/figures/score_margin_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved score_margin_histogram.png')